# Layer 5 — LLM Feedback Generation

Runs the full pipeline through difference analysis, then passes each movement's issues to the configured LLM to generate natural-language corrective feedback.

**Workflow**
1. Load + segment + align (same as notebooks 04–05)
2. Load thresholds and summarize issues per movement
3. Instantiate LLM client from `.env`
4. Generate feedback for movements 1–3 (spot-check)
5. Generate feedback for all 19 movements

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
POOMSAE           = "chon_ji"            # folder name under master_data/ and sample_videos/
POOMSAE_NAME      = "Chon-Ji poomsae"    # poomsae name used in the LLM system prompt
STUDENT_NAME      = "student2"           # filename (without extension) inside sample_videos/{POOMSAE}/
STUDENT_END_FRAME   = 934               # last frame of poomsae content
MASTER_START_FRAME  = 0                 # first frame of actual master poomsae (after ready stance)
STUDENT_START_FRAME = 0                 # first frame of actual student poomsae (after ready stance)
TOP_N             = 3                    # worst joints per movement
SPOT_CHECK_MOVS   = [1, 2, 3]           # movements to print individually first
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import os
import pathlib
import sys
import warnings

project_root = pathlib.Path(os.getcwd()).parent.parent
sys.path.insert(0, str(project_root))

from itf_analysis.normalization.normalizer import normalize_pose, extract_joint_angles
from itf_analysis.segmentation.keypose_matcher import load_master_keyposes
from itf_analysis.segmentation.segmentor import load_poses_from_json
from itf_analysis.segmentation.segmenter import segment_movements
from itf_analysis.alignment.dtw_aligner import (
    build_movement_segments, align_all_movements
)
from itf_analysis.feedback.difference_analyzer import (
    load_thresholds, summarize_movement_issues
)
from itf_analysis.feedback.llm_client import get_llm_client, LLMClientError
from itf_analysis.feedback.llm_feedback import (
    generate_movement_feedback, generate_full_feedback
)

sample_dir      = project_root / "itf_analysis" / "sample_videos"
poomsae_dir     = sample_dir / POOMSAE
master_dir      = project_root / "itf_analysis" / "master_data" / POOMSAE
keyposes_path   = str(master_dir / "keyposes_angles.json")
thresholds_path = str(master_dir / "thresholds.json")

MASTER_POSES_PATH  = str(poomsae_dir / "master_poses.json")
STUDENT_POSES_PATH = str(poomsae_dir / f"{STUDENT_NAME}_poses.json")

print("Imports OK")

## Step 1 — Load, segment, and align

In [19]:
def _load_angles(path):
    frames = load_poses_from_json(path)
    fps = 1000.0 / (frames[1].timestamp_ms - frames[0].timestamp_ms)
    angles = [
        (f.frame_index, extract_joint_angles(norm))
        for f in frames
        if (norm := normalize_pose(f.landmarks)) is not None
    ]
    return angles, fps

master_angles, master_fps = _load_angles(MASTER_POSES_PATH)
student_angles, student_fps = _load_angles(STUDENT_POSES_PATH)

keyposes = load_master_keyposes(keyposes_path)
keyposes_no_src = [{k: v for k, v in kp.items() if k != "source_frame"} for kp in keyposes]

with warnings.catch_warnings(record=True):
    warnings.simplefilter("always")
    master_boundaries = segment_movements(master_angles, keyposes, master_fps, start_frame=MASTER_START_FRAME)

with warnings.catch_warnings(record=True):
    warnings.simplefilter("always")
    student_boundaries = segment_movements(
        student_angles, keyposes_no_src, student_fps,
        start_frame=STUDENT_START_FRAME, end_frame=STUDENT_END_FRAME
    )

master_segments  = build_movement_segments(master_angles, master_boundaries)
student_segments = build_movement_segments(student_angles, student_boundaries)
all_results      = align_all_movements(master_segments, student_segments)

print(f"Master : {len(master_angles)} frames, {len(master_boundaries)} boundaries")
print(f"Student: {len(student_angles)} frames, {len(student_boundaries)} boundaries")
print(f"Aligned: {len(all_results)} / 19 movements")

Master : 1151 frames, 19 boundaries
Student: 964 frames, 19 boundaries
Aligned: 19 / 19 movements


## Step 2 — Summarize issues per movement

In [20]:
thresholds = load_thresholds(thresholds_path)

summaries = {
    mov: summarize_movement_issues(result, thresholds, top_n=TOP_N)
    for mov, result in all_results.items()
}

# Quick overview
print(f"{'Mov':>4} {'mean_rms':>9} {'#issues':>8}  Top issue")
print("-" * 55)
for mov in sorted(summaries):
    s = summaries[mov]
    top = f"{s.issues[0].joint} ({s.issues[0].direction})" if s.issues else "—"
    print(f"{mov:>4} {s.mean_rms:>9.1f} {len(s.issues):>8}  {top}")

 Mov  mean_rms  #issues  Top issue
-------------------------------------------------------
   1      11.8        0  —
   2      22.0        3  left_elbow (too_large)
   3      17.4        3  right_elbow (too_small)
   4      26.4        3  right_elbow (too_large)
   5      24.1        3  hip_line_angle (too_large)
   6      12.9        2  right_hip (too_small)
   7      21.1        3  right_elbow (too_large)
   8      17.8        1  left_elbow (too_small)
   9      13.9        1  left_elbow (too_small)
  10      41.3        3  right_elbow (too_large)
  11      25.3        3  right_elbow (too_large)
  12      14.1        0  —
  13      19.2        3  left_elbow (too_small)
  14      25.0        3  left_elbow (too_small)
  15      29.0        3  left_elbow (too_small)
  16      15.1        1  right_elbow (too_large)
  17      14.1        1  right_knee (too_small)
  18      13.9        0  —
  19      14.7        2  left_knee (too_small)


## Step 3 — Instantiate LLM client

Provider is read from `LLM_PROVIDER` in `.env`.  
Change the value there to switch between `gemini`, `claude`, or `deepseek` — no code change needed.

In [21]:
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

client = get_llm_client()
print(f"LLM client: {type(client).__name__}")

LLM client: GroqClient


## Step 4 — Spot-check: feedback for movements 1–3

In [22]:
kp_map = {kp["movement_index"]: kp for kp in keyposes}

for mov in SPOT_CHECK_MOVS:
    s = summaries.get(mov)
    if s is None:
        print(f"\nMovement {mov}: not aligned (skipped)")
        continue

    kp = kp_map.get(mov, {})
    itf  = kp.get("itf_name", "")
    name = kp.get("movement_name", "").replace("_", " ")

    print(f"\n{'━'*60}")
    print(f"Movement {mov}: {name}")
    print(f"  {itf}  |  mean RMS: {s.mean_rms:.1f}°")
    print(f"Issues: ", end="")
    if s.issues:
        print(", ".join(f"{i.joint} {i.direction} ({i.severity:.1f}×)" for i in s.issues))
    else:
        print("none")
    print()

    feedback = generate_movement_feedback(
        s, client,
        movement_name=name,
        itf_name=itf,
    )
    print(f"Feedback:")
    print(f"  {feedback}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Movement 1: left walking high block
  Ap Kubi Olgul Makgi (L)  |  mean RMS: 11.8°
Issues: none

Feedback:
  No significant issues detected for this movement.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Movement 2: right walking middle punch
  Ap Kubi Kaunde Jirugi (R)  |  mean RMS: 22.0°
Issues: left_elbow too_large (1.3×), hip_line_angle too_large (1.2×), right_elbow too_small (1.1×)

Feedback:
  Your left elbow needs to be slightly bent to maintain a more natural alignment. Your hip rotation should be more controlled, avoiding over-rotation to improve balance and stability. Your right elbow should be straightened to deliver a more effective punch.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Movement 3: right walking high block
  Ap Kubi Olgul Makgi (R)  |  mean RMS: 17.4°
Issues: right_elbow too_small (1.3×), right_knee too_large (1.0×), left_elbow too_small (1.0×)

Feedback:
  Your righ

## Step 5 — Full feedback: all 19 movements

In [ ]:
all_feedback = generate_full_feedback(summaries, client, keyposes=keyposes, poomsae_name=POOMSAE_NAME)

for mov in sorted(all_feedback):
    kp  = kp_map.get(mov, {})
    itf = kp.get("itf_name", f"Movement {mov}")
    print(f"\nMovement {mov:>2}: {itf}")
    print(f"  {all_feedback[mov]}")